# CNN + Transformer Hybrids


| Model | Inspiration | Idea |
|-------|-------------|------|
| **CNN_ViT_Hybrid** | ViT-style | CNN local features → spatial tokens → Transformer encoder |
| **EfficientFaceLite** | EfficientFace | Lightweight CNN + cross-spatial attention |
| **POSTERLite** | POSTER / POSTER++ | Multi-scale CNN pyramid → token fusion → Transformer |


Same data split, augmentation, and Wandb project (`CNN`) as `model_experiment_CNN.ipynb`.


In [ ]:
# Cell 1: Imports, device, Wandb
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms

import wandb

KAGGLE_DATA = Path(
    "/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/"
)
DATA_DIR = KAGGLE_DATA if KAGGLE_DATA.exists() else Path("data")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} | DATA_DIR: {DATA_DIR}")

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


def setup_wandb() -> None:
    api_key = os.environ.get("WANDB_API_KEY")
    if api_key is None:
        try:
            from kaggle_secrets import UserSecretsClient
            api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
        except Exception:
            pass
    if api_key:
        os.environ["WANDB_API_KEY"] = api_key
        # key= avoids interactive login hang on Run All
        wandb.login(key=api_key, relogin=False)
    else:
        print("WANDB_API_KEY not set — wandb runs may fail; training still works.")


setup_wandb()

In [ ]:
# Cell 2: Dataset
class FERDataset(Dataset):
    IMAGE_SIZE = 48
    NUM_PIXELS = IMAGE_SIZE * IMAGE_SIZE

    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.has_labels = "emotion" in self.dataframe.columns

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(self, idx: int):
        pixel_str = self.dataframe.loc[idx, "pixels"]
        pixels = np.fromstring(pixel_str, sep=" ", dtype=np.float32)
        if pixels.size != self.NUM_PIXELS:
            raise ValueError(f"Expected {self.NUM_PIXELS} pixels, got {pixels.size}")
        image = torch.from_numpy(pixels.reshape(self.IMAGE_SIZE, self.IMAGE_SIZE) / 255.0).unsqueeze(0)
        if self.transform is not None:
            image = self.transform(image)
        if self.has_labels:
            return image, int(self.dataframe.loc[idx, "emotion"])
        return image

In [ ]:
# Cell 3: Splits, augmentation, loaders
BATCH_SIZE = 64
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

TRAIN_AUGMENT = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=12),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.92, 1.08)),
])

df = pd.read_csv(DATA_DIR / "train.csv")
print(f"Loaded {len(df)} samples")

full_dataset = FERDataset(df)
train_size = int(len(full_dataset) * TRAIN_RATIO)
val_size = int(len(full_dataset) * VAL_RATIO)
test_size = len(full_dataset) - train_size - val_size
generator = torch.Generator().manual_seed(RANDOM_SEED)
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size], generator=generator
)
print(f"Train {len(train_dataset)} | Val {len(val_dataset)} | Test {len(test_dataset)}")


class TransformedSubset(Dataset):
    def __init__(self, subset: Dataset, transform=None):
        self.subset = subset
        self.transform = transform

    def __len__(self) -> int:
        return len(self.subset)

    def __getitem__(self, idx: int):
        image, label = self.subset[idx]
        if self.transform is not None:
            image = self.transform(image)
        return image, label


def build_augmented_loaders(batch_size: int = BATCH_SIZE):
    kwargs = {"num_workers": 0, "pin_memory": torch.cuda.is_available()}
    train_aug = TransformedSubset(train_dataset, TRAIN_AUGMENT)
    train_loader = DataLoader(train_aug, batch_size=batch_size, shuffle=True, **kwargs)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, **kwargs)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, **kwargs)
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = build_augmented_loaders()

# Class weights for imbalanced emotions (used in Cell 9 final ViT run)
_num_classes = 7
_train_labels = [train_dataset[i][1] for i in range(len(train_dataset))]
_class_counts = np.bincount(_train_labels, minlength=_num_classes).astype(np.float32)
CLASS_WEIGHTS = torch.tensor(
    len(_train_labels) / (_num_classes * np.maximum(_class_counts, 1.0)),
    dtype=torch.float32,
)
print(f"Class counts (train): {_class_counts.astype(int)}")
print(f"Class weights: {CLASS_WEIGHTS.numpy().round(3)}")

In [ ]:
# Cell 4: Training utilities
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        bs = labels.size(0)
        running_loss += loss.item() * bs
        correct += (outputs.argmax(1) == labels).sum().item()
        total += bs
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        bs = labels.size(0)
        running_loss += loss.item() * bs
        correct += (outputs.argmax(1) == labels).sum().item()
        total += bs
    return running_loss / total, correct / total


def sanity_check(model, dataloader, device, num_images=2, epochs=50, lr=1e-2):
    model = model.to(device).train()
    images, labels = next(iter(dataloader))
    images, labels = images[:num_images].to(device), labels[:num_images].to(device)
    criterion, optimizer = nn.CrossEntropyLoss(), optim.Adam(model.parameters(), lr=lr)
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        if epoch % 10 == 0 or epoch == 1:
            acc = (outputs.argmax(1) == labels).float().mean().item()
            print(f"  Epoch {epoch:3d} | Loss: {loss.item():.6f} | Acc: {acc * 100:.1f}%")
    acc = (outputs.argmax(1) == labels).float().mean().item()
    passed = loss.item() < 0.05 and acc == 1.0
    print("Sanity Check Passed" if passed else f"Sanity Check FAILED (loss={loss.item():.4f}, acc={acc:.2f})")
    return passed


def run_training(
    model, train_loader, val_loader, test_loader, *, run_name, project="CNN",
    epochs=30, learning_rate=1e-3, model_name="Hybrid", evaluate_test=False,
    extra_config=None, class_weights=None,
):
    torch.manual_seed(RANDOM_SEED)
    model = model.to(device)
    if class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    else:
        criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    config = {
        "model": model_name, "epochs": epochs, "batch_size": train_loader.batch_size,
        "learning_rate": learning_rate, "optimizer": "Adam", "random_seed": RANDOM_SEED,
        "augmentation": "flip+rotation+affine", "device": str(device),
        "class_weighted_loss": class_weights is not None,
    }
    if extra_config:
        config.update(extra_config)
    wandb.init(project=project, name=run_name, config=config, reinit=True)
    print(f"Training {run_name} for {epochs} epochs on {device}")
    best_val_acc = 0.0
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        best_val_acc = max(best_val_acc, val_acc)
        wandb.log({"epoch": epoch, "train_loss": train_loss, "train_accuracy": train_acc,
                   "val_loss": val_loss, "val_accuracy": val_acc})
        print(f"Epoch {epoch:3d}/{epochs} | Train {train_acc*100:.2f}% | Val {val_acc*100:.2f}%")
    wandb.log({"best_val_accuracy": best_val_acc})
    wandb.finish()
    print(f"Finished {run_name} | Best Val: {best_val_acc*100:.2f}%")
    return {"best_val_accuracy": best_val_acc, "final_train_accuracy": train_acc}, model


def evaluate_final_test(model, test_loader, *, run_name, project="CNN", model_name="Hybrid"):
    criterion = nn.CrossEntropyLoss()
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    wandb.init(project=project, name=run_name, config={"model": model_name, "split": "test"}, reinit=True)
    wandb.log({"test_loss": test_loss, "test_accuracy": test_acc})
    wandb.finish()
    print(f"Final test | {run_name} | Acc: {test_acc*100:.2f}%")
    return test_loss, test_acc


WANDB_PROJECT = "CNN"
EPOCHS = 40           # comparison training
EPOCHS_FINAL = 40     # final run (when enabled in Cell 9)
HYBRID_LR = 1e-3      # best ensemble used lr=1e-3 (ViT + POSTERLite)
DROPOUT = 0.3

CKPT_DIR = Path("/kaggle/working/checkpoints") if Path("/kaggle/working").exists() else Path("checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)


def hybrid_ckpt_path(arch_name: str) -> Path:
    return CKPT_DIR / f"{arch_name}.pt"

In [ ]:
# Cell 5: Shared blocks — Transformer encoder on spatial tokens
class ConvBNReLU(nn.Sequential):
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, padding=1):
        super().__init__(nn.Conv2d(in_ch, out_ch, kernel, stride, padding, bias=False),
                         nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))


class TransformerEncoder(nn.Module):
    def __init__(self, d_model=128, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        return self.encoder(tokens)


def tokens_from_feature_map(feat: torch.Tensor) -> torch.Tensor:
    """(B, C, H, W) -> (B, H*W, C)"""
    return feat.flatten(2).transpose(1, 2)

In [ ]:
# Cell 6: Hybrid architectures
NUM_CLASSES = 7


class CNN_ViT_Hybrid(nn.Module):
    """CNN stem extracts local features; Transformer models global spatial context."""

    def __init__(self, d_model=128, nhead=4, num_layers=2, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            ConvBNReLU(1, 32), nn.MaxPool2d(2),
            ConvBNReLU(32, 64), nn.MaxPool2d(2),
            ConvBNReLU(64, d_model), nn.MaxPool2d(2),  # (B, D, 6, 6)
        )
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.pos_embed = nn.Parameter(torch.zeros(1, 37, d_model))  # 36 patches + cls
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.transformer = TransformerEncoder(d_model, nhead, num_layers, dropout)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Dropout(dropout), nn.Linear(d_model, NUM_CLASSES))

    def forward(self, x):
        feat = self.stem(x)
        tokens = tokens_from_feature_map(feat)
        cls = self.cls_token.expand(x.size(0), -1, -1)
        tokens = torch.cat([cls, tokens], dim=1) + self.pos_embed
        tokens = self.transformer(tokens)
        return self.head(tokens[:, 0])


class CrossSpatialAttention(nn.Module):
    """EfficientFace-style cross-spatial attention (horizontal + vertical)."""

    def __init__(self, channels: int):
        super().__init__()
        self.conv_h = nn.Conv2d(channels, channels, (1, 7), padding=(0, 3), groups=channels, bias=False)
        self.conv_w = nn.Conv2d(channels, channels, (7, 1), padding=(3, 0), groups=channels, bias=False)
        self.act = nn.Sigmoid()

    def forward(self, x):
        return x * self.act(self.conv_h(x) + self.conv_w(x))


class DepthwiseSeparable(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, 3, stride, 1, groups=in_ch, bias=False),
            nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class EfficientFaceLite(nn.Module):
    """Lightweight CNN + cross-spatial attention + compact Transformer head."""

    def __init__(self, d_model=96, nhead=4, num_layers=1, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(ConvBNReLU(1, 32), nn.MaxPool2d(2))
        self.block1 = DepthwiseSeparable(32, 64, stride=1)
        self.attn1 = CrossSpatialAttention(64)
        self.pool1 = nn.MaxPool2d(2)
        self.block2 = DepthwiseSeparable(64, d_model, stride=1)
        self.attn2 = CrossSpatialAttention(d_model)
        self.pool2 = nn.MaxPool2d(2)
        self.transformer = TransformerEncoder(d_model, nhead, num_layers, dropout)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Dropout(dropout), nn.Linear(d_model, NUM_CLASSES))

    def forward(self, x):
        x = self.pool1(self.attn1(self.block1(self.stem(x))))
        x = self.pool2(self.attn2(self.block2(x)))
        tokens = tokens_from_feature_map(x)
        tokens = self.transformer(tokens)
        return self.head(tokens.mean(dim=1))


class POSTERLite(nn.Module):
    """POSTER-inspired pyramid: multi-scale CNN features fused via Transformer (no landmark branch)."""

    def __init__(self, d_model=96, nhead=4, num_layers=2, dropout=0.3, grid=4):
        super().__init__()
        self.grid = grid
        self.branch_s1 = nn.Sequential(ConvBNReLU(1, 32), nn.MaxPool2d(2))           # 24x24
        self.branch_s2 = nn.Sequential(ConvBNReLU(32, 64), nn.MaxPool2d(2))          # 12x12
        self.branch_s3 = nn.Sequential(ConvBNReLU(64, d_model), nn.MaxPool2d(2))     # 6x6
        self.proj1 = nn.Conv2d(32, d_model, 1, bias=False)
        self.proj2 = nn.Conv2d(64, d_model, 1, bias=False)
        self.pool_tokens = nn.AdaptiveAvgPool2d(grid)
        n_tokens = 3 * grid * grid
        self.pos_embed = nn.Parameter(torch.zeros(1, n_tokens, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.transformer = TransformerEncoder(d_model, nhead, num_layers, dropout)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Dropout(dropout), nn.Linear(d_model, NUM_CLASSES))

    def _scale_tokens(self, feat, proj=None):
        if proj is not None:
            feat = proj(feat)
        return tokens_from_feature_map(self.pool_tokens(feat))

    def forward(self, x):
        f1 = self.branch_s1(x)
        f2 = self.branch_s2(f1)
        f3 = self.branch_s3(f2)
        tokens = torch.cat([
            self._scale_tokens(f1, self.proj1),
            self._scale_tokens(f2, self.proj2),
            self._scale_tokens(f3, None),
        ], dim=1)
        tokens = tokens + self.pos_embed
        tokens = self.transformer(tokens)
        return self.head(tokens.mean(dim=1))


HYBRID_ARCHITECTURES = {
    "CNN_ViT_Hybrid": CNN_ViT_Hybrid,
    "EfficientFaceLite": EfficientFaceLite,
    "POSTERLite": POSTERLite,
}

for name, cls in HYBRID_ARCHITECTURES.items():
    m = cls()
    print(f"{name}: {sum(p.numel() for p in m.parameters()):,} params | out {m(torch.randn(2,1,48,48)).shape}")
    del m


def save_hybrid_ckpt(model: nn.Module, arch_name: str) -> Path:
    import shutil
    path = hybrid_ckpt_path(arch_name)
    torch.save({"arch": arch_name, "state_dict": model.state_dict()}, path)
    backup_dir = CKPT_DIR.parent / "checkpoints_backup"
    backup_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path, backup_dir / path.name)
    print(f"Saved checkpoint → {path}")
    print(f"Backup copy   → {backup_dir / path.name}")
    return path


def load_hybrid_ckpt(arch_name: str) -> nn.Module | None:
    path = hybrid_ckpt_path(arch_name)
    if not path.exists():
        backup = CKPT_DIR.parent / "checkpoints_backup" / f"{arch_name}.pt"
        if backup.exists():
            path = backup
            print(f"Using backup checkpoint: {path}")
        else:
            print(f"Missing checkpoint: {hybrid_ckpt_path(arch_name)}")
            return None
    payload = torch.load(path, map_location=device)
    model = HYBRID_ARCHITECTURES[payload["arch"]]()
    model.load_state_dict(payload["state_dict"])
    print(f"Loaded checkpoint ← {path}")
    return model.to(device)


def list_hybrid_ckpts(arch_names: list[str] | None = None) -> None:
    names = arch_names or list(HYBRID_ARCHITECTURES.keys())
    print("=== Checkpoints on disk ===")
    for arch in names:
        p = hybrid_ckpt_path(arch)
        b = CKPT_DIR.parent / "checkpoints_backup" / f"{arch}.pt"
        status = "OK" if p.exists() else ("BACKUP" if b.exists() else "MISSING")
        print(f"  {status}: {p}")

In [ ]:
# Cell 7: Experiment toggles — IDLE (best model already trained)
#
# BEST MODEL (saved Kaggle version):
#   lr=1e-3 | ViT test 59.23% | Ensemble test 62.15%
#   Checkpoints: CNN_ViT_Hybrid.pt + POSTERLite.pt
#
# Re-run ensemble only (same session, both .pt on disk):
#   Cell 9:  RUN_VIT_FINAL = False
#   Cell 10: RUN_ENSEMBLE_TEST = True
#
# HP tuning → model_experiment_HP.ipynb

RUN_HYBRID_SANITY_CHECK = False
RUN_HYBRID_COMPARISON = False

SELECTED_HYBRID_ARCH = "CNN_ViT_Hybrid"
HYBRID_VAL_ACC = 0.6122     # ViT 40-ep val (lr=1e-3)
HYBRID_TEST_ACC = 0.5923    # ViT standalone test
ENSEMBLE_TEST_ACC = 0.6215  # BEST — equal-weight ensemble

RUN_HYBRID_SANITY_CHECK = False
RUN_HYBRID_COMPARISON = False

SELECTED_HYBRID_ARCH = "CNN_ViT_Hybrid"
HYBRID_VAL_ACC = 0.6122     # ViT 40-ep final (lr=1e-3)
HYBRID_TEST_ACC = 0.5923    # ViT standalone test
ENSEMBLE_TEST_ACC = 0.6215  # ViT + POSTERLite ensemble test

In [ ]:
# Cell 8: Architecture comparison (40 epochs, val only) — train ONE per session
HYBRID_ARCHS_TO_TRAIN = []

RECORDED_HYBRID_RESULTS = [
    {"run_name": "CNN_ViT_Hybrid", "best_val_accuracy": 0.6085, "final_train_accuracy": 0.5969, "epochs": 30, "note": "comparison"},
    {"run_name": "EfficientFaceLite", "best_val_accuracy": 0.5896, "final_train_accuracy": 0.5885, "epochs": 40},
    {"run_name": "POSTERLite", "best_val_accuracy": 0.6064, "final_train_accuracy": 0.6250, "epochs": 40, "note": "comparison"},
    {"run_name": "CNN_ViT_Hybrid_Final_lr5e-4", "best_val_accuracy": 0.6125, "test_accuracy": 0.6125, "epochs": 40, "note": "final lr=5e-4"},
    {"run_name": "POSTERLite_EnsembleCkpt", "best_val_accuracy": 0.6152, "epochs": 40, "note": "retrained same session"},
    {"run_name": "Ensemble_ViT_POSTERLite", "test_accuracy": 0.6116, "note": "lr5e-4 ViT + POSTER — this session"},
    {"run_name": "Ensemble_ViT_POSTERLite_best", "test_accuracy": 0.6215, "note": "BEST — lr=1e-3 ViT + POSTER (prior session)"},
]

hybrid_results = list(RECORDED_HYBRID_RESULTS)
# hybrid_model is set in Cell 9 only — never assign None here

if RUN_HYBRID_SANITY_CHECK:
    arch = HYBRID_ARCHS_TO_TRAIN[0] if HYBRID_ARCHS_TO_TRAIN else "CNN_ViT_Hybrid"
    # val_loader has no augmentation — required for stable overfit sanity check
    sanity_check(HYBRID_ARCHITECTURES[arch](), val_loader, device)
else:
    print("Skipping sanity check.")

if RUN_HYBRID_COMPARISON:
    for arch_name in HYBRID_ARCHS_TO_TRAIN:
        print(f"\n=== {arch_name} ===")
        model = HYBRID_ARCHITECTURES[arch_name]()
        metrics, trained = run_training(
            model, train_loader, val_loader, test_loader,
            run_name=arch_name, project=WANDB_PROJECT, epochs=EPOCHS,
            learning_rate=HYBRID_LR, model_name=arch_name, evaluate_test=False,
            extra_config={"family": "CNN+Transformer", "dropout": DROPOUT},
        )
        hybrid_results.append({
            "run_name": arch_name,
            "best_val_accuracy": metrics["best_val_accuracy"],
            "final_train_accuracy": metrics["final_train_accuracy"],
        })
        if arch_name == SELECTED_HYBRID_ARCH or metrics["best_val_accuracy"] > (HYBRID_VAL_ACC or 0):
            hybrid_model = trained
else:
    print("Skipping comparison (RUN_HYBRID_COMPARISON=False).")

if hybrid_results:
    df_hybrid = pd.DataFrame(hybrid_results).sort_values("best_val_accuracy", ascending=False)
    print("\n=== CNN+Transformer comparison (val) ===")
    display(df_hybrid)
    if RUN_HYBRID_COMPARISON:
        SELECTED_HYBRID_ARCH = df_hybrid.iloc[0]["run_name"]
        HYBRID_VAL_ACC = df_hybrid.iloc[0]["best_val_accuracy"]
    print(f"Leader: {df_hybrid.iloc[0]['run_name']} | Val: {df_hybrid.iloc[0]['best_val_accuracy']*100:.2f}%")
else:
    print("No hybrid runs yet.")

In [ ]:
# Cell 9: Train ViT + save checkpoint (+ optional standalone test)
RUN_VIT_FINAL = False             # BEST ViT already saved — do not retrain
SKIP_VIT_IF_CKPT_EXISTS = True
USE_CLASS_WEIGHTS = False
RUN_VIT_STANDALONE_TEST = False

VIT_ARCH = "CNN_ViT_Hybrid"
FINAL_RUN_NAME = "CNN_ViT_Hybrid_Final"

if RUN_VIT_FINAL:
    vit_ckpt = hybrid_ckpt_path(VIT_ARCH)
    if SKIP_VIT_IF_CKPT_EXISTS and vit_ckpt.exists():
        print(f"ViT checkpoint exists — skipping train: {vit_ckpt}")
        vit_model = load_hybrid_ckpt(VIT_ARCH)
    else:
        cw = CLASS_WEIGHTS if USE_CLASS_WEIGHTS else None
        print(
            f"=== {VIT_ARCH}: {EPOCHS_FINAL} epochs | "
            f"class weights={'ON' if USE_CLASS_WEIGHTS else 'OFF'} ==="
        )
        vit_model = HYBRID_ARCHITECTURES[VIT_ARCH]()
        vit_metrics, vit_model = run_training(
            vit_model,
            train_loader,
            val_loader,
            test_loader,
            run_name=FINAL_RUN_NAME,
            project=WANDB_PROJECT,
            epochs=EPOCHS_FINAL,
            learning_rate=HYBRID_LR,
            model_name=VIT_ARCH,
            class_weights=cw,
            extra_config={
                "family": "CNN+Transformer",
                "final": True,
                "class_weighted_loss": USE_CLASS_WEIGHTS,
            },
        )
        HYBRID_VAL_ACC = vit_metrics["best_val_accuracy"]
        SELECTED_HYBRID_ARCH = VIT_ARCH
        ckpt_path = save_hybrid_ckpt(vit_model, VIT_ARCH)
        print(f"ViT trained | val: {HYBRID_VAL_ACC * 100:.2f}% | checkpoint: {ckpt_path}")

    if RUN_VIT_STANDALONE_TEST and vit_model is not None:
        _, HYBRID_TEST_ACC = evaluate_final_test(
            vit_model,
            test_loader,
            run_name=f"{VIT_ARCH}_FinalTest",
            project=WANDB_PROJECT,
            model_name=VIT_ARCH,
        )
        print(f"ViT standalone test: {HYBRID_TEST_ACC * 100:.2f}%")
elif HYBRID_TEST_ACC is not None:
    print(
        f"ViT recorded | val: {HYBRID_VAL_ACC * 100:.2f}% | "
        f"test: {HYBRID_TEST_ACC * 100:.2f}% (RUN_VIT_FINAL=False)"
    )
else:
    print("Cell 9 skipped (RUN_VIT_FINAL=False).")

list_hybrid_ckpts([VIT_ARCH])

In [ ]:
# Cell 10: Train POSTERLite + ensemble test
RUN_TRAIN_POSTER_CKPT = False      # BEST POSTERLite already saved
SKIP_POSTER_IF_CKPT_EXISTS = True
RUN_ENSEMBLE_TEST = False          # completed — 62.15% test

ENSEMBLE_ARCHS = ["CNN_ViT_Hybrid", "POSTERLite"]
POSTER_ARCH = "POSTERLite"

list_hybrid_ckpts(ENSEMBLE_ARCHS)


@torch.no_grad()
def ensemble_predict(models: list[nn.Module], images: torch.Tensor) -> torch.Tensor:
    # Average raw logits from each model, then argmax once (standard logit ensemble)
    return sum(m(images) for m in models) / len(models)


if RUN_TRAIN_POSTER_CKPT:
    poster_ckpt = hybrid_ckpt_path(POSTER_ARCH)
    if SKIP_POSTER_IF_CKPT_EXISTS and poster_ckpt.exists():
        print(f"POSTERLite checkpoint exists — skipping train: {poster_ckpt}")
    else:
        print(f"Training {POSTER_ARCH} ({EPOCHS_FINAL} epochs) → save checkpoint...")
        poster = HYBRID_ARCHITECTURES[POSTER_ARCH]()
        poster_metrics, poster = run_training(
            poster, train_loader, val_loader, test_loader,
            run_name="POSTERLite_EnsembleCkpt", project=WANDB_PROJECT, epochs=EPOCHS_FINAL,
            learning_rate=HYBRID_LR, model_name=POSTER_ARCH,
            extra_config={"purpose": "ensemble_checkpoint"},
        )
        save_hybrid_ckpt(poster, POSTER_ARCH)
        print(f"POSTERLite trained | val: {poster_metrics['best_val_accuracy'] * 100:.2f}%")
else:
    print("Skipping POSTERLite training (RUN_TRAIN_POSTER_CKPT=False).")

list_hybrid_ckpts(ENSEMBLE_ARCHS)

if RUN_ENSEMBLE_TEST:
    models = []
    for arch in ENSEMBLE_ARCHS:
        m = load_hybrid_ckpt(arch)
        if m is None:
            raise RuntimeError(
                f"Missing {arch} checkpoint at {hybrid_ckpt_path(arch)}.\n"
                f"  ViT: set RUN_VIT_FINAL=True in Cell 9 (same session).\n"
                f"  POSTERLite: set RUN_TRAIN_POSTER_CKPT=True in Cell 10.\n"
                f"  Note: /kaggle/working/checkpoints/ is empty in a new Kaggle session."
            )
        models.append(m)
    correct, total = 0, 0
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = ensemble_predict(models, images).argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    ens_acc = correct / total
    print(f"\n=== ENSEMBLE RESULT ===")
    print(f"Models: {' + '.join(ENSEMBLE_ARCHS)}")
    print(f"Test accuracy: {ens_acc * 100:.2f}%")
    if HYBRID_TEST_ACC is not None:
        print(f"ViT alone was: {HYBRID_TEST_ACC * 100:.2f}%")
    wandb.init(project=WANDB_PROJECT, name="Ensemble_ViT_POSTERLite_FinalTest", reinit=True)
    wandb.log({"test_accuracy": ens_acc, "models": ENSEMBLE_ARCHS})
    wandb.finish()
elif ENSEMBLE_TEST_ACC is not None:
    print(f"Ensemble recorded | test: {ENSEMBLE_TEST_ACC * 100:.2f}% (RUN_ENSEMBLE_TEST=False)")
else:
    print("Ensemble test not run (RUN_ENSEMBLE_TEST=False).")